In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as Sum
from os.path import basename, abspath

In [2]:
problem_name = basename(abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 23:24:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
sales = spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .format("csv")\
    .load("files/sales.txt")

In [5]:
sales.show()

+-----------+-----------+
|   category|total_price|
+-----------+-----------+
|       Food|       5000|
|Electronics|      20000|
|       Food|       3000|
|   Clothing|      10000|
|Electronics|      15000|
|       Food|       2500|
+-----------+-----------+



In [6]:
sales.createOrReplaceTempView("sales")

In [7]:
result = sales.groupBy(col("category")).agg(Sum(col("total_price")).alias("total_sales"))

In [8]:
result.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|       Food|      10500|
|Electronics|      35000|
|   Clothing|      10000|
+-----------+-----------+



In [9]:
spark.sql(
    """
    select category, sum(total_price) as total_sales
    from sales
    group by category
    """
).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|       Food|      10500|
|Electronics|      35000|
|   Clothing|      10000|
+-----------+-----------+



In [10]:
spark.stop()